<a href="https://colab.research.google.com/github/jegedeolakunle21-art/Energy-Consumption-Predictor/blob/main/Energy_Consumption_Predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

Saving energy_model.pkl to energy_model (1).pkl


In [3]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.3 MB/s eta 0:00:00


In [4]:
!pip install pyngrok

In [5]:
%%writefile app.py
import streamlit as st
import sys
import warnings
from datetime import datetime

import joblib
import pandas as pd
import streamlit as st

warnings.filterwarnings("ignore")

# NOTE: energy_model.pkl was trained/pickled with scikit-learn 1.6.1.
# scikit-learn pickles are not guaranteed compatible across versions, so
# requirements.txt pins scikit-learn==1.6.1 to match. Using a different
# installed version can raise AttributeErrors (e.g. missing internal
# attributes on ColumnTransformer/SimpleImputer) or silently mispredict.

MODEL_PATH = "energy_model.pkl"

st.set_page_config(page_title="Energy Consumption Predictor", page_icon="⚡", layout="centered")


@st.cache_resource
def load_model(path: str):
    return joblib.load(path)


try:
    model = load_model(MODEL_PATH)
except FileNotFoundError:
    st.error(f"Model file not found at '{MODEL_PATH}'. Place energy_model.pkl next to app.py.")
    st.stop()
except Exception as e:
    st.error(f"Failed to load model: {e}")
    st.stop()

st.title("⚡ Energy Consumption Predictor")
st.write("Enter the building and environment conditions below to predict energy consumption.")

with st.form("prediction_form"):
    st.subheader("Time")
    c1, c2, c3 = st.columns(3)
    with c1:
        hour = st.number_input("Hour", min_value=0, max_value=23, value=datetime.now().hour)
        year = st.number_input("Year", min_value=2000, max_value=2100, value=datetime.now().year)
    with c2:
        minute = st.number_input("Minute", min_value=0, max_value=59, value=0)
        month = st.number_input("Month", min_value=1, max_value=12, value=datetime.now().month)
    with c3:
        day = st.number_input("Day", min_value=1, max_value=31, value=datetime.now().day)
        day_of_week = st.selectbox(
            "Day of week",
            ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
        )

    st.subheader("Environment")
    c1, c2 = st.columns(2)
    with c1:
        temperature = st.number_input("Temperature", value=22.0, step=0.5)
        humidity = st.number_input("Humidity", value=50.0, step=1.0)
    with c2:
        occupancy = st.number_input("Occupancy", min_value=0, value=10)
        square_footage = st.number_input("Square Footage", min_value=0.0, value=1000.0, step=10.0)

    st.subheader("Building systems")
    c1, c2 = st.columns(2)
    with c1:
        hvac_usage = st.selectbox("HVAC usage", ["Off", "On"])
    with c2:
        lighting_usage = st.selectbox("Lighting Usage", ["Off", "On"])

    st.subheader("Other factors")
    c1, c2, c3 = st.columns(3)
    with c1:
        holiday = st.selectbox("Holiday", ["No", "Yes"])
    with c2:
        holiday_analysis = st.number_input("Holiday Analysis", value=0.0, step=1.0)
    with c3:
        renewable_energy = st.number_input("Renewable Energy", value=0.0, step=1.0)

    c1, c2 = st.columns(2)
    with c1:
        net_energy = st.number_input("Net Energy", value=0.0, step=1.0)
    with c2:
        temp_occupancy = st.number_input(
            "Temp Occupancy", value=float(temperature) * float(occupancy), step=1.0
        )

    submitted = st.form_submit_button("Predict energy consumption")

if submitted:
    input_df = pd.DataFrame(
        [
            {
                "Hour": hour,
                "Temperature": temperature,
                "Humidity": humidity,
                "Occupancy": occupancy,
                "HVAC usage": hvac_usage,
                "Square Footage": square_footage,
                "Lighting Usage": lighting_usage,
                "Day of week": day_of_week,
                "Holiday": holiday,
                "Holiday Analysis": holiday_analysis,
                "Renewable Energy": renewable_energy,
                "Net Energy": net_energy,
                "Temp Occupancy": temp_occupancy,
                "Year": year,
                "Month": month,
                "Day": day,
                "Minute": minute,
            }
        ]
    )

    try:
        prediction = model.predict(input_df)[0]
        st.success(f"Predicted energy consumption: **{prediction:,.2f}**")
        with st.expander("Show input data sent to the model"):
            st.dataframe(input_df.T.rename(columns={0: "value"}))
    except Exception as e:
        st.error(f"Prediction failed: {e}")

st.caption(
    "Model: scikit-learn Pipeline (ColumnTransformer → RandomForestRegressor). "
    "Trained feature set is fixed; all fields above map directly to the model's expected inputs."
)

Writing app.py


In [6]:
!ls

 app.py  'energy_model (1).pkl'   energy_model.pkl   sample_data


In [7]:
!ngrok authtoken 3FJAIhnHCW5DjI1tMIMGOkTlOLe_vnwC2U47LMggYGVNhVaW

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [8]:
!ls ngrok

ls: cannot access 'ngrok': No such file or directory


In [9]:
pip install pyngrok

In [10]:
!ngrok

ngrok - tunnel local ports to public URLs and inspect traffic

USAGE:
  ngrok [command] [flags]

COMMANDS: 
  api             CLI to api.ngrok.com
  completion      generates shell completion code for bash or zsh
  config          update or migrate ngrok's configuration file
  credits         prints author and licensing information
  help            help about any command
  http            start an HTTP tunnel
  service         run and control ngrok as a background service
  start           start endpoints in the config file by name
  tcp             start a TCP tunnel
  tls             start a TLS endpoint
  update          update ngrok to the latest version
  version         print the version string

EXAMPLES: 
# forward http traffic from assigned public URL to local port 80
ngrok http 80
# port 8080 available at baz.ngrok.dev
ngrok http --url baz.ngrok.dev 8080
# tunnel arbitrary TCP traffic to port 22
ngrok tcp 22
# secure your app with oauth
ngrok http 80 --oauth=google --oauth-al

In [11]:
from pyngrok import ngrok

In [12]:
!streamlit run app.py &>/dev/null&

In [13]:
!pgrep streamlit

5759


In [14]:
!ps -p 22493 -p 35676 -o pid,lstart,cmd

    PID                  STARTED CMD


In [15]:
public_url = ngrok.connect(addr='8501')

In [16]:
print(public_url)

NgrokTunnel: "https://refract-flagstone-scarf.ngrok-free.dev" -> "http://localhost:8501"


In [18]:
import joblib

# Load the uploaded model to define the 'model' variable in your session
model = joblib.load("energy_model.pkl")
print("Model loaded successfully:", model)

Model loaded successfully: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Hour', 'Temperature', 'Humidity', 'Occupancy', 'Square Footage',
       'Holiday Analysis', 'Renewable Energy', 'Net Energy', 'Temp Occupancy',
       'Year', 'Month', 'Day', 'Minute'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                          

In [19]:
import os

print(os.path.exists("energy_model.pkl"))

True


In [20]:
from google.colab import files

files.download("energy_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>